## Tratamento dos dados

Nesse notebook os dados são tratados de forma a tornar possível a análise

In [ ]:
import pandas as pd
import unicodedata
from utils.load_csv import load_csv
from utils.load_shp import load_shp
from utils.save_shp import save_shp

In [2]:
#df_tse_2026 = load_csv('resultados_2026.csv')

df_observasampa = load_csv('observasampa.csv')

gdf_distritos = load_shp('distritos.shp')

1 - Filtros das bases de dados

In [3]:
# Indicadores socioeconômicos do município de São Paulo por distrito
# Seleciona as variáveis de interesse, filtra apenas os distritos e o período de 2024

variaveis = ["01.03.02 Quantidade de famílias beneficiárias do Programa Bolsa Família",
             "População total"]

df_observasampa_filtrado = df_observasampa[df_observasampa["Nome"].isin(variaveis)
                                           & df_observasampa["Região"].str.endswith("(Distrito)")
                                           & (df_observasampa["Período"] == 2024)].copy()

df_observasampa_filtrado["Resultado"] = pd.to_numeric(df_observasampa_filtrado["Resultado"])

df_observasampa_filtrado = (df_observasampa_filtrado
                            .pivot(index="Região", columns="Nome", values="Resultado")
                            .rename(columns={"01.03.02 Quantidade de famílias beneficiárias do Programa Bolsa Família": "familias_pbf",
                                             "População total": "pop_total"})
                            .rename_axis(columns=None)     
                            .reset_index()
                            .rename(columns={"Região": "distrito"}))

df_observasampa_filtrado["distrito"] = df_observasampa_filtrado["distrito"].str.replace(r"\s*\(Distrito\)$", "", regex=True)

In [4]:
df_observasampa_filtrado

,distrito,familias_pbf,pop_total
0,Alto de Pinheiros,717,36987
1,Anhanguera,14600,75967
2,Aricanduva,12412,89499
3,Artur Alvim,16091,93634
4,Barra Funda,1522,36499
...,...,...,...
91,Vila Matilde,11127,103089
92,Vila Medeiros,20782,111745
93,Vila Prudente,10687,106187
94,Vila Sônia,11084,125416


In [5]:
# Malha territorial dos limites administrativos dos distritos do estado de São Paulo
# Seleciona as variáveis de interesse, filtra apenas os distritos da cidade de São Paulo

gdf_distritos_filtrado = gdf_distritos.loc[gdf_distritos["CD_MUN"] == "3550308",
                                           ["CD_DIST", "NM_DIST", "geometry"]].reset_index(drop=True)

In [6]:
gdf_distritos_filtrado

,CD_DIST,NM_DIST,geometry
0,355030801,Água Rasa,"POLYGON ((-46.57207 -23.57948, -46.57212 -23.5..."
1,355030802,Alto de Pinheiros,"POLYGON ((-46.69581 -23.55327, -46.69604 -23.5..."
2,355030803,Anhanguera,"POLYGON ((-46.76434 -23.4369, -46.76431 -23.43..."
3,355030804,Aricanduva,"POLYGON ((-46.51779 -23.58627, -46.51786 -23.5..."
4,355030805,Artur Alvim,"POLYGON ((-46.48574 -23.55441, -46.48574 -23.5..."
...,...,...,...
91,355030892,Vila Medeiros,"POLYGON ((-46.57783 -23.49527, -46.57794 -23.4..."
92,355030893,Vila Prudente,"POLYGON ((-46.5748 -23.60107, -46.57499 -23.60..."
93,355030894,Vila Sônia,"POLYGON ((-46.72791 -23.61149, -46.72798 -23.6..."
94,355030895,São Domingos,"POLYGON ((-46.7467 -23.5075, -46.74671 -23.507..."


2 - Indicador de famílias beneficiárias do PBF por mil habitantes por distrito

In [7]:
df_observasampa_filtrado['pbf_mil_hab'] = (df_observasampa_filtrado['familias_pbf'] / df_observasampa_filtrado['pop_total']) * 1000

df_observasampa_filtrado

,distrito,familias_pbf,pop_total,pbf_mil_hab
0,Alto de Pinheiros,717,36987,19.385189
1,Anhanguera,14600,75967,192.188714
2,Aricanduva,12412,89499,138.683114
3,Artur Alvim,16091,93634,171.849969
4,Barra Funda,1522,36499,41.699773
...,...,...,...,...
91,Vila Matilde,11127,103089,107.935861
92,Vila Medeiros,20782,111745,185.977001
93,Vila Prudente,10687,106187,100.643205
94,Vila Sônia,11084,125416,88.377878


3 - Junção das bases de dados

In [8]:
def normaliza(s):
    s = str(s).strip().upper()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    return " ".join(s.split())

gdf_distritos_filtrado["chave"] = gdf_distritos_filtrado["NM_DIST"].apply(normaliza)
df_observasampa_filtrado["chave"] = df_observasampa_filtrado["distrito"].apply(normaliza)

gdf_final = gdf_distritos_filtrado.merge(df_observasampa_filtrado.drop(columns="distrito"),
                                         on="chave",
                                         how="left",
                                         indicator=True)

In [9]:
gdf_final

,CD_DIST,NM_DIST,geometry,chave,familias_pbf,pop_total,pbf_mil_hab,_merge
0,355030801,Água Rasa,"POLYGON ((-46.57207 -23.57948, -46.57212 -23.5...",AGUA RASA,7377,85993,85.786052,both
1,355030802,Alto de Pinheiros,"POLYGON ((-46.69581 -23.55327, -46.69604 -23.5...",ALTO DE PINHEIROS,717,36987,19.385189,both
2,355030803,Anhanguera,"POLYGON ((-46.76434 -23.4369, -46.76431 -23.43...",ANHANGUERA,14600,75967,192.188714,both
3,355030804,Aricanduva,"POLYGON ((-46.51779 -23.58627, -46.51786 -23.5...",ARICANDUVA,12412,89499,138.683114,both
4,355030805,Artur Alvim,"POLYGON ((-46.48574 -23.55441, -46.48574 -23.5...",ARTUR ALVIM,16091,93634,171.849969,both
...,...,...,...,...,...,...,...,...
91,355030892,Vila Medeiros,"POLYGON ((-46.57783 -23.49527, -46.57794 -23.4...",VILA MEDEIROS,20782,111745,185.977001,both
92,355030893,Vila Prudente,"POLYGON ((-46.5748 -23.60107, -46.57499 -23.60...",VILA PRUDENTE,10687,106187,100.643205,both
93,355030894,Vila Sônia,"POLYGON ((-46.72791 -23.61149, -46.72798 -23.6...",VILA SONIA,11084,125416,88.377878,both
94,355030895,São Domingos,"POLYGON ((-46.7467 -23.5075, -46.74671 -23.507...",SAO DOMINGOS,9983,89226,111.884428,both


4 - Configuração final da base de dados

In [10]:
gdf_final = (gdf_final
             .rename(columns={"CD_DIST": "cd_dist", "NM_DIST": "distrito"})
             .drop(columns=["chave", "_merge"]))

gdf_final

,cd_dist,distrito,geometry,familias_pbf,pop_total,pbf_mil_hab
0,355030801,Água Rasa,"POLYGON ((-46.57207 -23.57948, -46.57212 -23.5...",7377,85993,85.786052
1,355030802,Alto de Pinheiros,"POLYGON ((-46.69581 -23.55327, -46.69604 -23.5...",717,36987,19.385189
2,355030803,Anhanguera,"POLYGON ((-46.76434 -23.4369, -46.76431 -23.43...",14600,75967,192.188714
3,355030804,Aricanduva,"POLYGON ((-46.51779 -23.58627, -46.51786 -23.5...",12412,89499,138.683114
4,355030805,Artur Alvim,"POLYGON ((-46.48574 -23.55441, -46.48574 -23.5...",16091,93634,171.849969
...,...,...,...,...,...,...
91,355030892,Vila Medeiros,"POLYGON ((-46.57783 -23.49527, -46.57794 -23.4...",20782,111745,185.977001
92,355030893,Vila Prudente,"POLYGON ((-46.5748 -23.60107, -46.57499 -23.60...",10687,106187,100.643205
93,355030894,Vila Sônia,"POLYGON ((-46.72791 -23.61149, -46.72798 -23.6...",11084,125416,88.377878
94,355030895,São Domingos,"POLYGON ((-46.7467 -23.5075, -46.74671 -23.507...",9983,89226,111.884428
